# GARCH Model in Python: A 36-Day Shock Half-Life

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/quant-finance/garch_volatility.ipynb)

Build the likelihood in NumPy and SciPy, check it against `arch`, then compare causal forecasts. The S&P 500 close snapshot covers 3 January 2000 to 30 September 2026. It is embedded below so the results can be reproduced offline. The initial fit and historical illustrations use the full sample. Only the walk-forward comparison measures future forecast loss.

Companion post: https://sesen.ai/blog/garch-model-python

## Packages

The reference library is `arch` 8.0.0. Colab may need to install it. The reference run used NumPy 2.2.4, SciPy 1.15.2 and pandas 2.2.3. The code below installs only the missing reference package. The remaining imports are standard Colab packages.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('arch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'arch==8.0.0'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
from scipy.stats import norm, t
from statsmodels.tsa.stattools import acf
from matplotlib.animation import FuncAnimation, PillowWriter
from pathlib import Path

## Pinned data

The raw Yahoo Finance `^GSPC` adjusted-close snapshot was fetched with `yfinance.download('^GSPC', start='2000-01-01', end='2026-10-01', auto_adjust=True)`. The end date is exclusive. The hash verifies the snapshot before parsing. Returns are percentage log returns, with a fixed zero-mean model.

In [ ]:
import base64
import gzip
import hashlib
import io

SNAPSHOT = '...'
raw = gzip.decompress(base64.b64decode(SNAPSHOT))
assert hashlib.sha256(raw).hexdigest() == '3d8e4334c257a087c477445e4e753c26114d8cebb434b6d3fa19962243c7861f'
closes = pd.read_csv(io.BytesIO(raw), index_col='date', parse_dates=True).close
series = 100 * np.log(closes).diff().dropna()
r = series.to_numpy()
print(f'{len(closes)} closes, {len(r)} returns, through {closes.index[-1].date()}')

## Variance recursion and likelihood

GARCH updates tomorrow's variance with a baseline, the last squared shock and stored variance. `lfilter` computes the same recursion in compiled code. Both starts use constrained SLSQP. The Student-t density is scaled to unit innovation variance.

In [ ]:
"""Zero-mean GARCH(1,1), fitted directly with NumPy and SciPy.

Returns are percentage log returns. The initial variance uses arch's backcast,
so the reference comparison evaluates the same likelihood.
"""

import numpy as np
from scipy.optimize import minimize
from scipy.signal import lfilter
from scipy.special import gammaln


def backcast(returns):
    n = min(75, len(returns))
    weights = 0.94 ** np.arange(n)
    return float(weights @ np.square(returns[:n]) / weights.sum())


def variance_path(params, returns, initial=None):
    omega, alpha, beta = params[:3]
    initial = backcast(returns) if initial is None else initial
    lagged_squares = np.r_[initial, np.square(returns[:-1])]
    # lfilter evaluates h[t] = omega + alpha*r[t-1]**2 + beta*h[t-1].
    return lfilter([1.0], [1.0, -beta], omega + alpha * lagged_squares,
                   zi=[beta * initial])[0]


def negative_loglikelihood(params, returns, initial, student=False):
    h = variance_path(params, returns, initial)
    if np.any(h <= 0) or not np.all(np.isfinite(h)):
        return 1e100
    if not student:
        return float(0.5 * np.sum(np.log(2 * np.pi * h) + returns**2 / h))
    nu = params[3]
    constant = gammaln((nu + 1) / 2) - gammaln(nu / 2)
    constant -= 0.5 * np.log(np.pi * (nu - 2))
    log_density = constant - 0.5 * np.log(h)
    log_density -= (nu + 1) / 2 * np.log1p(returns**2 / (h * (nu - 2)))
    return float(-log_density.sum())


def fit_garch(returns, student=False, start=None):
    returns = np.asarray(returns, dtype=float)
    if returns.ndim != 1 or len(returns) < 75 or not np.isfinite(returns).all():
        raise ValueError("Supply at least 75 finite, one-dimensional returns.")
    initial = backcast(returns)
    scale = float(np.mean(returns**2))
    if scale <= 0:
        raise ValueError("Returns must have positive second moment.")
    starts = [np.array([0.02 * scale, 0.08, 0.90]),
              np.array([0.05 * scale, 0.15, 0.80])]
    if student:
        starts = [np.r_[s, 8.0] for s in starts]
    if start is not None:
        starts.insert(0, np.asarray(start))
    bounds = [(1e-9, 10 * scale), (0, 1), (0, 1)]
    if student:
        bounds.append((2.05, 200))
    constraint = {"type": "ineq", "fun": lambda p: 0.999999 - p[1] - p[2]}
    results = []
    for s in starts:
        history = [s.copy()]
        result = minimize(negative_loglikelihood, s,
                          args=(returns, initial, student), method="SLSQP",
                          bounds=bounds, constraints=constraint,
                          callback=lambda p: history.append(p.copy()),
                          options={"ftol": 1e-8, "maxiter": 300})
        if result.success and np.isfinite(result.fun):
            result.history = history + [result.x.copy()]
            results.append(result)
    if not results:
        raise RuntimeError("All constrained likelihood fits failed.")
    return min(results, key=lambda result: result.fun)


def forecast_path(params, last_return, last_variance, horizon=20):
    omega, alpha, beta = params[:3]
    first = omega + alpha * last_return**2 + beta * last_variance
    persistence = alpha + beta
    long_run = omega / (1 - persistence)
    return long_run + persistence ** np.arange(horizon) * (first - long_run)


def ewma_path(returns, decay=0.94):
    initial = backcast(returns)
    return lfilter([1 - decay], [1, -decay], np.r_[initial, returns[:-1]**2],
                   zi=[decay * initial])[0]


def qlike(proxy, forecast):
    # Equivalent rankings to proxy/h - log(proxy/h) - 1; accepts zero returns.
    return np.log(forecast) + proxy / forecast


## Reference fit and forecasts

Matching the mean model, initial variance and units makes the reference comparison meaningful. The checks cover both likelihood fits, the entire Gaussian variance path at identical parameters and a 60-step forecast. Small differences in optimiser output are permitted.

In [ ]:
fit = fit_garch(r)
fit_t = fit_garch(r, student=True)
reference = arch_model(r, mean='Zero', p=1, q=1, rescale=False).fit(disp='off', backcast=backcast(r), options={'ftol': 1e-9})
reference_t = arch_model(r, mean='Zero', p=1, q=1, dist='StudentsT', rescale=False).fit(disp='off', backcast=backcast(r))
h = variance_path(fit.x, r)
fixed = reference.model.fix(fit.x)
np.testing.assert_allclose(fit.x, reference.params, rtol=1e-3, atol=1e-5)
np.testing.assert_allclose(fit_t.x, reference_t.params, rtol=1e-3, atol=1e-4)
np.testing.assert_allclose(h, fixed.conditional_volatility**2, rtol=1e-7)
np.testing.assert_allclose(forecast_path(fit.x, r[-1], h[-1], 60), fixed.forecast(horizon=60).variance.iloc[-1], rtol=1e-7)
persistence = fit.x[1] + fit.x[2]
print(pd.DataFrame({'ours': fit.x, 'arch': reference.params.to_numpy()}, index=['omega', 'alpha', 'beta']))
print('Shock half-life (trading days):', np.log(.5) / np.log(persistence))

## Walk-forward comparison

At each close, fit on the last 1,260 returns, refitting every 20 trading days. Each forecast uses data through that origin only. The 20-day target is the average squared return over the next 20 trading days, matched to the average of the forecast variance path. EWMA uses λ = 0.94. The rolling baseline is the zero-mean second moment over 20 returns. The same origins are used at both horizons. Approximate 95% intervals use a 40-lag Bartlett HAC estimate, because overlapping targets create dependence.

In [ ]:
def paired_hac(diff, lag=40):
    """95% normal interval using a Bartlett HAC variance for the mean."""
    centred = diff - diff.mean()
    n = len(diff)
    variance = float(centred @ centred / n)
    for k in range(1, lag + 1):
        variance += 2 * (1 - k / (lag + 1)) * float(centred[k:] @ centred[:-k] / n)
    se = np.sqrt(max(variance, 0) / n)
    return [float(diff.mean() - 1.96 * se), float(diff.mean() + 1.96 * se)]

def walk_forward(returns, dates, window=1260, every=20):
    ewma = ewma_path(returns)
    records, warm = [], None
    for origin in range(window - 1, len(returns) - 20):
        if (origin - window + 1) % every == 0:
            train = returns[origin - window + 1:origin + 1]
            fit = fit_garch(train, start=warm)
            warm = fit.x
            h = variance_path(warm, train)[-1]
        else:
            h = warm[0] + warm[1] * returns[origin - 1]**2 + warm[2] * h
        garch = forecast_path(warm, returns[origin], h)
        ewma_next = 0.94 * ewma[origin] + 0.06 * returns[origin]**2
        rolling = float(np.mean(returns[origin - 19:origin + 1]**2))
        for horizon in (1, 20):
            proxy = float(np.mean(returns[origin + 1:origin + horizon + 1]**2))
            record = {"origin": str(dates[origin].date()), "horizon": horizon,
                      "proxy": proxy, "garch": float(garch[:horizon].mean()),
                      "ewma": float(ewma_next), "rolling": rolling}
            records.append(record)
    return pd.DataFrame(records)

In [ ]:
table = walk_forward(r, series.index)
results = {'gaussian': fit.x.tolist(), 'student_t': fit_t.x.tolist(), 'scores': {}}
for horizon, subset in table.groupby('horizon'):
    losses = {name: qlike(subset.proxy.to_numpy(), subset[name].to_numpy()) for name in ['garch', 'ewma', 'rolling']}
    summary = {name: float(value.mean()) for name, value in losses.items()}
    summary['garch_minus_ewma_ci95'] = paired_hac(losses['garch'] - losses['ewma'])
    results['scores'][str(horizon)] = summary
    print(horizon, summary)
# Detect any accidental look-ahead in the first forecast when later data change.
changed = r.copy()
changed[1260:] *= 3
prefix_check = walk_forward(changed[:1300], series.index[:1300]).iloc[0]
np.testing.assert_allclose(prefix_check[['garch','ewma','rolling']].astype(float), table.iloc[0][['garch','ewma','rolling']].astype(float), rtol=1e-10)

## Figures and optimisation animation

Every panel uses the fitted models or measured walk-forward losses. The historical forecast illustrations use full-sample parameters to show mean reversion. The animation shows the best likelihood reached among feasible accepted SLSQP iterates. Different vertical scales make the calm and stressed forecasts readable.

In [ ]:
OUT = Path('garch_figures')
OUT.mkdir(exist_ok=True)
SLUG = 'garch-model-python'
TEAL, AMBER, NAVY = '#258d84', '#d49636', '#354e68'
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
def save(fig, name):
    fig.savefig(OUT / f'{SLUG}-{name}.webp', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)

def build():
    series = 100 * np.log(closes).diff().dropna()
    r, dates = series.to_numpy(), series.index
    p = np.array(results["gaussian"])
    pt = np.array(results["student_t"])
    h = variance_path(p, r)
    ht = variance_path(pt, r)

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    for ax, values, title, colour in zip(axes, [r, r**2],
                                       ["Returns", "Squared returns"], [NAVY, TEAL]):
        values_acf = acf(values, nlags=40, fft=True)[1:]
        ax.vlines(np.arange(1, 41), 0, values_acf, color=colour, lw=2)
        ax.scatter(np.arange(1, 41), values_acf, color=colour, s=12)
        ax.axhspan(-1.96 / np.sqrt(len(r)), 1.96 / np.sqrt(len(r)), color=NAVY, alpha=.1)
        ax.axhline(0, color=NAVY, lw=.6)
        ax.set(xlabel="Lag (trading days)", title=title)
    axes[0].set_ylabel("Sample autocorrelation")
    fig.tight_layout()
    save(fig, "acf")

    fig, ax = plt.subplots(figsize=(11, 4.3))
    ax.plot(dates, np.abs(r), color=NAVY, alpha=.22, lw=.7, label="Absolute daily return")
    ax.plot(dates, np.sqrt(h), color=TEAL, lw=1.2, label="GARCH conditional volatility")
    ax.set(ylabel="Daily return / volatility (%)", title="S&P 500, January 2000 to September 2026")
    for date, label, ypos in [("2000-03-10", "Dot-com peak", 10.7),
                              ("2008-09-15", "Lehman", 12.5),
                              ("2020-02-20", "COVID", 10.7)]:
        ax.axvline(pd.Timestamp(date), color=AMBER, ls=":", lw=1)
        ax.text(pd.Timestamp(date), ypos, label, ha="center", fontsize=9,
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": .9})
    ax.legend(loc="upper right")
    fig.tight_layout()
    save(fig, "volatility")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, (date, label, colour) in zip(axes, [("2017-11-01", "Calm: 1 November 2017", TEAL),
                                              ("2020-03-16", "Stressed: 16 March 2020", AMBER)]):
        i = dates.get_loc(pd.Timestamp(date))
        forecasts = forecast_path(p, r[i], h[i], 180)
        ax.plot(np.arange(1, 181), forecasts, color=colour, lw=2)
        ax.axhline(p[0] / (1 - p[1] - p[2]), color=NAVY, ls="--", label="Long-run variance")
        ax.set(xlabel="Horizon (trading days)", ylabel="Forecast daily variance (%²)", title=label)
        ax.legend()
    fig.tight_layout()
    save(fig, "forecasts")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, horizon in zip(axes, (1, 20)):
        scores = results["scores"][str(horizon)]
        names = ["garch", "ewma", "rolling"]
        vals = [scores[name] for name in names]
        bars = ax.bar(["GARCH", "EWMA", "20-day rolling"], vals,
                      color=[TEAL, AMBER, NAVY], width=.65)
        for bar, value in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, value+.025, f"{value:.3f}", ha="center")
        ax.set(ylim=(0, 1.4), ylabel="Mean QLIKE loss (lower is better)",
               title=f"Next {horizon} trading {'day' if horizon == 1 else 'days (average variance)'}")
    fig.tight_layout()
    save(fig, "qlike")

    probs = (np.arange(len(r)) + .5) / len(r)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for ax, residuals, theory, label, colour in [
        (axes[0], r/np.sqrt(h), norm.ppf(probs), "Gaussian innovations", TEAL),
        (axes[1], r/np.sqrt(ht), t.ppf(probs, df=pt[3])*np.sqrt((pt[3]-2)/pt[3]),
         f"Student-t innovations (ν = {pt[3]:.2f})", AMBER)]:
        ax.scatter(theory, np.sort(residuals), s=9, color=colour, alpha=.65)
        limits = [min(theory.min(), residuals.min())-.4, max(theory.max(), residuals.max())+.4]
        ax.plot(limits, limits, ls="--", color=NAVY, lw=1)
        ax.set(xlabel="Theoretical quantile (unit variance)", title=label)
        ax.grid(alpha=.15)
    axes[0].set_ylabel("Standardised return quantile")
    fig.tight_layout()
    save(fig, "qq")

    fit = fit_garch(r)
    history = [point for point in fit.history
               if point[0] > 0 and min(point[1:]) >= 0 and sum(point[1:]) < 1]
    losses = np.array([negative_loglikelihood(point, r, backcast(r)) for point in history])
    best = np.minimum.accumulate(losses)
    # End shortly after the likelihood has converged, without a long flat tail.
    terminal = int(np.flatnonzero(best - best[-1] < 1e-5)[0])
    terminal = min(len(history), terminal + 2)
    history, losses, best = history[:terminal], losses[:terminal], best[:terminal]
    # Display best-so-far feasible iterates, retaining their real iteration numbers.
    chosen = [np.argmin(losses[:i+1]) for i in range(len(history))]
    points = [history[index] for index in chosen]
    segment = (dates >= "2008-08-01") & (dates <= "2009-02-01")
    fig, axes = plt.subplots(2, 1, figsize=(9, 5.8), gridspec_kw={"height_ratios": [2, 1]})
    axes[0].plot(dates[segment], np.abs(r[segment]), color=NAVY, alpha=.3,
                 label="Absolute return")
    line, = axes[0].plot([], [], color=TEAL, lw=2, label="Current fitted volatility")
    axes[0].set(ylabel="Daily return / volatility (%)", ylim=(0, max(abs(r[segment]))*1.12))
    axes[0].legend(loc="upper left")
    axes[1].plot(np.arange(len(best)), best, color=NAVY, alpha=.25)
    progress, = axes[1].plot([], [], color=TEAL, lw=2)
    dot, = axes[1].plot([], [], "o", color=AMBER)
    axes[1].set(xlabel="Feasible accepted iterate", ylabel="Best negative log-likelihood",
               xlim=(0, len(best)-1), ylim=(best[-1]-20, best[0]+50))
    fig.tight_layout()
    fig.subplots_adjust(top=.91)
    frames = np.linspace(0, len(points)-1, min(12, len(points)), dtype=int)
    def update(frame):
        i = frames[frame]
        line.set_data(dates[segment], np.sqrt(variance_path(points[i], r))[segment])
        progress.set_data(np.arange(i+1), best[:i+1])
        dot.set_data([i], [best[i]])
        axes[0].set_title(f"Likelihood optimisation: feasible iterate {i} of {len(points)-1}")
        return line, progress, dot
    animation = FuncAnimation(fig, update, frames=len(frames), interval=333, blit=False)
    gif = OUT / f"{SLUG}-fit.gif"
    animation.save(gif, writer=PillowWriter(fps=3), dpi=100)
    update(len(frames)-1)
    save(fig, "fit")
    from IPython.display import HTML, display
    display(HTML(animation.to_jshtml()))
    print("Five charts, optimisation still and GIF generated.")

build()

## Exercises

1. Compare five-year and ten-year fitting windows on a common evaluation period beginning after at least 2,520 returns. Run both protocols, retain the common origins, and compare paired loss differences.
2. Add Student-t GARCH to the walk-forward benchmark. An in-sample QQ improvement does not establish better future variance forecasts.
3. Estimate a constant mean using only each training window, and use those residuals consistently for every method.
4. Compare signed and squared residual autocorrelations after filtering.
5. Feed `forecast_path` into a volatility-targeting demonstration, converting percentage returns and variance into consistent decimal units first.

## References

[Engle (1982)](https://web.pdx.edu/~crkl/ec572/readings/Engle82.pdf), [Bollerslev (1986)](https://public.econ.duke.edu/~boller/Published_Papers/joe_86.pdf), [Patton (2011)](https://public.econ.duke.edu/~ap172/Patton_vol_proxies_JoE_2011.pdf), [RiskMetrics (1996)](https://www.msci.com/research-and-insights/paper/1996-riskmetrics-technical-document), [arch forecasting documentation](https://arch.readthedocs.io/en/latest/univariate/forecasting.html).